**Note:** in order to run the examples in this notebook you need a python environment with the following packages installed:

    numpypandas, matplotlib, tensorflow, ipykernel, palmerpenguins

# Workshop: Scientific Programming with Python using Jupyter Notebooks

<div>
<img src="https://www.python.org/static/img/python-logo.png" width="350"/>
<img src="https://www.gosmarter.ai/blog/glossary/what-is-jupyter/jupyter_hu_b6cacecfb7fece90.webp" width="250"/>
<img src="https://www.kim.uni-konstanz.de/fileadmin/user_upload/csm_bwHPC_logo_quer_transparent_bb75ad50f1.png" width="350"/>
</div> 

# Agenda

- **Introduction** <br>
    - Format of this Workshop
    - Project Jupyter 
    - Accessing the JupyterLab Server <br><br>
- **Part 1: Data Management, Data Visualization & Data Science Basics** <br>
    - Clustering
    - Dimensionality Reduction <br><br>
- **Part 2: Data Science and Machine Learning** <br>
    - *Part 2.0* Neural Network Basics
        - Artificial Neural Networks
        - Activation functions
    - *Part 2.1:* NNs with Pytorch
        - Regression task with NNs
        - Classification with NNs
        - Convolutional NNs
    - *Part 2.2:* NNs with Tensorflow
        - Denoising with NNs<br><br>
- **Part 3: Framework comparison exercise** <br>
    - Data Preparation
    - Tensorboard
    - NN with Pytorch
    - NN with Tensorflow
<br><br>
- **References**


# Part 2.2: Neural Networks with Tensorflow

### Image classification with a Neural Network

We will start with a basic classification task example from https://www.tensorflow.org/tutorials/keras/classification. For this we use the Fashion MNIST dataset. It consists of gray scale images with a resolution of 28x28 pixels of items of clothing by Zalando, each labeled as 1 of 10 different categories.

In [ ]:
# TensorFlow and tf.keras
import tensorflow as tf

# Helper libraries
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
CUDA_VISIBLE_DEVICES=""

#### Preparing the dataset

Now we load the dataset. Note that training data and testing data are stored in separate variables.

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist

(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

Classifications are stored as numbers within the datasets, not as words. To connect them, we create a list with the respective index corresponding to the number of the classification.

In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

Explore the formats of the loaded dataset fragments! (shape, number of entries, type of entries?)

In [ ]:
# your code goes here

The data needs to be preprocessed. As it is now the scanning would yield huge activation strengths for basically all neurons, because NNs typically are designed for normalized data with values rangig from 0 to 1. It's a bit like trying to count dark spots on the sun by looking directly at it with a basic unfiltered telescope in broad daylight - you'd become (at least temporarily) blinded because the intensity is far too high! (Note: we do vehemently suggest NOT to look at the sun without any suitable eye protection!)

In [ ]:
## Pixel values in the dataset range from 0 to 255. We scale them down to range from 0 to 1.

train_images = train_images / 255.0

## Processing of the test set has to be equivalent to the processing of the training set!

test_images = test_images / 255.0

Now for a quick look at an excerpt of the dataset.

In [ ]:
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i], cmap=plt.cm.binary)
    plt.xlabel(class_names[train_labels[i]])
plt.show()

#### Model setup and training

Now we have to set up our NN. NNs are made up of multiple connected layers, so we have to define our layer structue:

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input((28, 28,)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10)
])

We now created a simple model of sequential layers. This means we have a straightforward flow of info from one layer to the next, i.e. no layer has multiple inputs or outputs.

The first layer is our input layer. This one specifially turns our 28x28 input tensor into a one-dimesional tensor of size 784 (i.e. the tensor/array gets only rearranged, it's content stays the same).

The second layer is a Dense layer with 128 neurons (or alternatively called nodes). A dense layer means it is fully connected with it's respective input layer. You might have noticed the 'relu' argument for activation. This means we use the ReLU (rectified linear unit) function as activation function for our neurons. This basically means that if the sum of signals *x* the neuron receives is larger than 0, the sum becomes it's own signal (f(*x*) = *x*). Otherwise the neuron will assume a value of 0 (f(*x*) = 0).

The third layer here is our final layer, meaning our output layer. The value of a neuron in this layer corresponds with the determined (calculated / learned) probability for the respective category ID being the solution.

Now we have established the structure of our NN, but there remain some important things to be defined. Those will be added in the compile step of our model:

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

So let's disect this a bit.

The optimizer is the specific algorithm used for the adjustment of weights (i.e. connections between neurons) during the backpropagation when training.
With 'loss' we define the loss-function which determines how far the obtained result is from the intended ouput.
Using the 'metrics' keyword we tell the model which metric to calculate and display for us to observe training progression.
Another important setting taken here implicitly (through a default setting) is which compute unit (CPU or GPU) to use with our model. Your model then gets compiled (translated to machine code) for fast execution.

Now the model is prepared and training can begin. We pass our prepared image data and the respective target label IDs to the 'fit' function. We also set epochs to 10, which determines how often the model is going to train on the dataset.

In [ ]:
model.fit(train_images, train_labels, epochs=10)

In [ ]:
test_loss, test_acc = model.evaluate(test_images,  test_labels, verbose=2)

print('\nTest accuracy:', test_acc)

As you can see, we have a noticeably lower accuracy with our testing set compared to the training data. This means the model is overfitted to the training data and has learned intricacies special to our training set. Learning these intricacies however makes the model less powerful when predicting data that is not in the training set. There are a number of steps possible to minimize overfitting, e.g. using a larger dataset for training or having less trainable parameters (i.e. smaller layers), but proper training often takes time to find suitable parameters. 

We will also not focus on overfitting and it's obviation.

#### Prediction / Inference

Now that we have trained and tested our model, we want to use it to perform it's intended purpose, in our case to classify low resolution images of clothing articles. As our model will thus encounter data it has not been trained on, it will make a prediction based on what it has learned, i.e. it will infer from patterns known to it. This is why it is also called inference in the ML community.

As stated earlier, the output vector corresponds with respective predicted probabilities. We can turn it into actual probabilities by adding an extra layer to our model, which performs this conversion.

In [ ]:
probability_model = tf.keras.Sequential([model, 
                                         tf.keras.layers.Softmax()])

Because this extra layer only performs a simple conversion without any learnable parameters, we don't have to train our model again. Now we can use this extended model to make predictions for e.g. our test set. Afterwards we'll check some of the predicted probabilities our model made.

In [ ]:
predictions = probability_model.predict(test_images)

Now let's see what prediction our model made for an excerpt of our test data.

In [ ]:
# You can either define a set of image IDs or you can pick a number of random IDs.
test_ids = np.array([])
num_rand_images = 10

if test_ids.size == 0:
    num_test_images = test_images.shape[0]
    test_ids = np.random.randint(0, num_test_images, size=num_rand_images)
    del num_test_images

del num_rand_images

In [ ]:
class_ids = list(range(len(class_names)))
percentage = list(range(0, 101, 10))

In [ ]:
for i in test_ids:
    plt.figure(figsize=(6,3))
    plt.subplot(1,2,1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(test_images[i], cmap=plt.cm.binary)
    plt.xlabel(class_names[test_labels[i]])
    plt.title("Image + classification")
    plt.subplot(1,2,2)
    plt.bar(class_ids, predictions[i]*100)
    plt.xticks(class_ids, labels=class_names, rotation=80)
    plt.yticks(percentage)
    plt.title("Prediction")
    plt.show()
    print(predictions[i]*100, class_names[predictions[i].argmax()])

### Image denoising with a Convolutional Neural Network

In tasks such as computer vision or classification Convolutional Neural Networks often are employed. They work by moving convolution windows (also called filter) along the input data, which thereby extract features from the input data.

### Autoencoders

Autoencoders are a common ANN architecture. 

They work by compressing the input into a smaller representation and then reconstructing the original input from this compressed version. Here's a simple breakdown:

* **Encoder**: This part of the network compresses the input data into a smaller, dense representation (called a latent space or code).

* **Bottleneck**: This is the compressed form of the input data, capturing its most important features. It is simultaneously the output of the encoder and the input of the decoder.

* **Decoder**: This part of the network takes the compressed representation and reconstructs the original data as closely as possible.

Autoencoders are often used for purposes like noise reduction, data compression, dimensionality reduction and feature extraction. As these tasks show, autoencoders are very useful when defining a ground truth as training target is difficult.

<div>
<img src="https://images.squarespace-cdn.com/content/v1/62f1caa0a2cb083186ccce31/c3350b48-ff66-4e04-aa79-56cec8350ca2/autoencoder_architecture.png)" width="800"/>
</div> 

from https://www.saberhq.com/blog/autoencoders

#### Basic Autoencoder

The following example is based on https://www.tensorflow.org/tutorials/generative/autoencoder

#### Real World Data 3: Fashion MNIST

Fashion-MNIST is a dataset of Zalando's article images—consisting of a training set of 60,000 examples and a test set of 10,000 examples.

<div>
<img src="https://www.researchgate.net/publication/342801790/figure/fig2/AS:911232181735425@1594266090934/Sample-images-from-Fashion-MNIST-dataset.png" width="800"/>
</div> 

Kheradpisheh et al. Neural Process Lett 54, 1255–1273 (2022). https://doi.org/10.1007/s11063-021-10680-x

In [ ]:
import tensorflow.keras as keras

In [ ]:
# imports
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
#from tensorflow.keras import layers, losses
#from tensorflow.keras.datasets import fashion_mnist
#from tensorflow.keras.models import Model
import tensorflow as tf
import matplotlib.pyplot as plt

In [ ]:
# Load data
(x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()

x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

print (x_train.shape)
print (x_test.shape)

In [ ]:
n = 10
plt.figure(figsize=(20, 2))
for i in range(n):
    ax = plt.subplot(1, n, i + 1)
    plt.title("original image")
    plt.imshow(tf.squeeze(x_test[i]))
    plt.gray()
plt.show()

#### Denoising Images
An autoencoder can also be trained to remove noise from images. 

In the following section, you will create a noisy version of the Fashion MNIST dataset by applying random noise to each image. You will then train an autoencoder using the noisy image as input, and the original image as the target.

Let's reimport the dataset to omit the modifications made earlier. As we essentially train by de- and reconstruction of the images and don't perform a classification, we don't need the classification labels.

In [ ]:
(x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()

In [ ]:
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

x_train = x_train[..., tf.newaxis]
x_test = x_test[..., tf.newaxis]

print(x_train.shape)

Adding random noise to the images

In [ ]:
noise_factor = 0.2
x_train_noisy = x_train + noise_factor * tf.random.normal(shape=x_train.shape) 
x_test_noisy = x_test + noise_factor * tf.random.normal(shape=x_test.shape) 

x_train_noisy = tf.clip_by_value(x_train_noisy, clip_value_min=0., clip_value_max=1.)
x_test_noisy = tf.clip_by_value(x_test_noisy, clip_value_min=0., clip_value_max=1.)

In [ ]:
n = 10
plt.figure(figsize=(20, 2))
for i in range(n):
    ax = plt.subplot(1, n, i + 1)
    plt.title("original + noise")
    plt.imshow(tf.squeeze(x_test_noisy[i]))
    plt.gray()
plt.show()

##### Convolutional Autoencoder
In this example, you will train a convolutional autoencoder using Conv2D layers in the encoder, and Conv2DTranspose layers in the decoder. A Conv2DTranspose layer performs the reverse operation of a Conv2D layer.

In [ ]:
class Denoise(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.encoder = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(28, 28, 1)),
            tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same', strides=2),
            tf.keras.layers.Conv2D(8, (3, 3), activation='relu', padding='same', strides=2)])

        self.decoder = tf.keras.Sequential([
            tf.keras.layers.Conv2DTranspose(8, kernel_size=3, strides=2, activation='relu', padding='same'),
            tf.keras.layers.Conv2DTranspose(16, kernel_size=3, strides=2, activation='relu', padding='same'),
            tf.keras.layers.Conv2D(1, kernel_size=(3, 3), activation='sigmoid', padding='same')])

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

autoencoder = Denoise()

In this case the compression is a result of using a stride of 2, skipping every 2nd possible input configuration, along with a reduction of used filters in the convolution layers.

In [ ]:
autoencoder.compile(optimizer='adam', loss=tf.keras.losses.MeanSquaredError())

In [ ]:
autoencoder.fit(x_train_noisy, x_train,
                epochs=5,
                shuffle=True,
                validation_data=(x_test_noisy, x_test))

In [ ]:
autoencoder.encoder.summary()

In [ ]:
autoencoder.decoder.summary()

In [ ]:
encoded_imgs = autoencoder.encoder(x_test_noisy).numpy()
decoded_imgs = autoencoder.decoder(encoded_imgs).numpy()

In [ ]:
n = 10
plt.figure(figsize=(20, 4))
for i in range(n):

    # display original + noise
    ax = plt.subplot(3, n, i + 1)
    plt.title("original + noise")
    plt.imshow(tf.squeeze(x_test_noisy[i]))
    plt.gray()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # display reconstruction
    bx = plt.subplot(3, n, i + n + 1)
    plt.title("reconstructed")
    plt.imshow(tf.squeeze(decoded_imgs[i]))
    plt.gray()
    bx.get_xaxis().set_visible(False)
    bx.get_yaxis().set_visible(False)

    # display original
    cx = plt.subplot(3, n, i + 2*n + 1)
    plt.title("original")
    plt.imshow(tf.squeeze(x_test[i]))
    plt.gray()
    cx.get_xaxis().set_visible(False)
    cx.get_yaxis().set_visible(False)
    
plt.show()

### Resources

Introduction to Artificial Neural Networks:

- https://youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi&si=K6NmU277knsiknd7

- https://www.mzes.uni-mannheim.de/socialsciencedatalab/article/ann/

- https://www.geeksforgeeks.org/artificial-neural-networks-and-its-applications/

  

Autoencoders:

- https://towardsdatascience.com/introduction-to-autoencoders-7a47cf4ef14b

- https://www.tensorflow.org/tutorials/generative/autoencoder

- https://www.datacamp.com/tutorial/introduction-to-autoencoders

  

Convolutional Neural Networks:

- https://saturncloud.io/blog/a-comprehensive-guide-to-convolutional-neural-networks-the-eli5-way/

- https://www.youtube.com/watch?v=KuXjwB4LzSA&t=363s

- https://www.youtube.com/watch?v=py5byOOHZM8


Materials & Tutorials:

- https://www.tensorflow.org/tutorials/

- https://ki-kurs.org/ Online KI Kurs des Bundeswettbewerbs Künstliche Intelligenz

- https://huggingface.co/ Platform for tools for the creation of applications with machine learning.

# References

The content of this workshop is in parts based on and inspired by the following sources:

* Python Course of the AG Peter (Prof. Dr. Christine Peter, Kevin Savade, Dr. Oleksandra Kukharenko, Dr. Andrej Berg)
* Software Carpentry workshops (https://software-carpentry.org/lessons/)
* Online KI Kurs des Bundeswettbewerbs Künstliche Intelligenz (https://ki-kurs.org/)
* Real Python (https://realpython.com/)
* Intro to Autoencoders (https://www.tensorflow.org/tutorials/generative/autoencoder)
* Image classification of MNIST using TensorFlow (https://www.kaggle.com/code/viratkothari/image-classification-of-mnist-using-tensorflow)
* The bwHPC wiki (https://wiki.bwhpc.de/)